# Lab: escolhendo o melhor modelo com validação cruzada

Notebook de aula da página: https://insper.github.io/ML/pt/2026.2/classes/validation/

Rode as células em ordem. Em cada passo, responda o **Preveja** com a dupla *antes* de rodar. Nas **Simulações**, escreva o experimento na célula vazia; as soluções e a leitura de cada resultado estão na página da aula.

## Passo 1 — carregar e corrigir o rótulo

**Preveja.** No conjunto do scikit-learn, `target = 1` é benigno. Se você calcular o "recall" sem mexer nisso, o que exatamente estaria medindo?

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
X = data.data                     # 30 medidas numéricas de cada tumor
y = 1 - data.target               # o scikit-learn codifica benigno = 1; invertemos para 1 = maligno
print(X.shape)
print(y.value_counts())

(569, 30)
target
0    357
1    212
Name: count, dtype: int64


## Passo 2 — trancar o conjunto de teste

**Preveja.** Esta é a única divisão feita à mão no lab inteiro. Por que o `stratify` importa aqui, com 37% de malignos? Até que passo o `X_test` não pode ser tocado?

In [2]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
print(X_train.shape, X_test.shape)
print(f"fração maligna: treino {y_train.mean():.3f} · teste {y_test.mean():.3f}")

(455, 30) (114, 30)
fração maligna: treino 0.374 · teste 0.368


## Passo 3 — um baseline, para os números significarem algo

**Preveja.** Um modelo que sempre responde "benigno" vai tirar que acurácia, com 37% de malignos? E que recall?

In [3]:
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

dummy = DummyClassifier(strategy='most_frequent')          # sempre diz "benigno"
for metric in ['accuracy', 'recall']:
    s = cross_val_score(dummy, X_train, y_train, cv=cv, scoring=metric)
    print(f"{metric:8s} {s.mean():.3f}")

accuracy 0.626
recall   0.000


## Passo 4 — CV 5-fold com várias métricas

**Preveja.** Os cinco folds vão dar o mesmo recall? Se variarem, você diria que a diferença vem do modelo ou de quais tumores caíram em cada fold?

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))

res = cross_validate(model, X_train, y_train, cv=cv, return_train_score=True,
                     scoring=['accuracy', 'precision', 'recall', 'f1', 'roc_auc'])
folds = pd.DataFrame(res).drop(columns=['fit_time', 'score_time'])
print(folds[['train_recall', 'test_accuracy', 'test_precision', 'test_recall', 'test_roc_auc']].round(3))
print(folds.agg(['mean', 'std']).T.round(3))

   train_recall  test_accuracy  test_precision  test_recall  test_roc_auc
0         0.978          0.956           1.000        0.882         0.995
1         0.963          0.967           0.943        0.971         0.999
2         0.963          1.000           1.000        1.000         1.000
3         0.978          0.978           1.000        0.941         0.987
4         0.971          0.967           0.943        0.971         0.998
                  mean    std
test_accuracy    0.974  0.017
train_accuracy   0.988  0.004
test_precision   0.977  0.031
train_precision  0.997  0.004
test_recall      0.953  0.045
train_recall     0.971  0.007
test_f1          0.964  0.023
train_f1         0.984  0.005
test_roc_auc     0.996  0.005
train_roc_auc    0.998  0.001


**Simulação: a sorte da divisão, medida.** A aula afirmou que um único split é uma loteria. Meça isso nos dados do lab: para 20 sementes, compare a acurácia de um único hold-out 80/20 com a média de uma CV 5-fold. Que faixa cada um percorre?

In [5]:
hold, cvm = [], []
for seed in range(20):
    a, b, c, d = train_test_split(X_train, y_train, test_size=0.2, stratify=y_train, random_state=seed)
    hold.append(model.fit(a, c).score(b, d))
    cvm.append(cross_val_score(model, X_train, y_train,
                               cv=StratifiedKFold(5, shuffle=True, random_state=seed)).mean())
print(f"hold-out 80/20: {min(hold):.3f} a {max(hold):.3f} (desvio {np.std(hold):.3f})")
print(f"CV 5-fold:      {min(cvm):.3f} a {max(cvm):.3f} (desvio {np.std(cvm):.3f})")

hold-out 80/20: 0.934 a 1.000 (desvio 0.016)
CV 5-fold:      0.969 a 0.980 (desvio 0.003)


## Passo 5 — uma média mais estável: CV repetida

**Preveja.** Repetir a CV 5-fold dez vezes com embaralhamentos diferentes muda mais a média ou a confiança que você tem nela?

In [6]:
from sklearn.model_selection import RepeatedStratifiedKFold

rcv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)
rec = cross_val_score(model, X_train, y_train, cv=rcv, scoring='recall')
print(f"{len(rec)} escores · recall {rec.mean():.3f} ± {rec.std():.3f} · "
      f"de {rec.min():.3f} a {rec.max():.3f}")

50 escores · recall 0.947 ± 0.037 · de 0.853 a 1.000


## Passo 6 — ajustar `C` com uma busca em grade

**Preveja.** `C` pequeno é penalidade forte. Com `C = 0,001`, o recall de treino e o de validação vão ficar os dois baixos, os dois altos ou longe um do outro? E com `C = 1000`?

Lembre que o `C` faz o papel de $1/\alpha$, o inverso da força da penalidade $\alpha$ do [Ridge](https://insper.github.io/ML/pt/2026.2/classes/gradient-descent-regularization/#regularizacao) (a menos de um fator constante): `C` pequeno = penalidade forte.

In [7]:
from sklearn.model_selection import GridSearchCV

grid = GridSearchCV(model, {'logisticregression__C': np.logspace(-3, 3, 13)},
                    cv=cv, scoring='recall', return_train_score=True)
grid.fit(X_train, y_train)

table = pd.DataFrame(grid.cv_results_)[['param_logisticregression__C', 'mean_train_score',
                                        'mean_test_score', 'std_test_score']]
table.columns = ['C', 'recall treino', 'recall val', 'desvio val']
print(table.round(3).to_string(index=False))
print(f"melhor: C = {grid.best_params_['logisticregression__C']:g} · recall {grid.best_score_:.3f}")

       C  recall treino  recall val  desvio val
   0.001          0.694       0.700       0.088
   0.003          0.843       0.841       0.084
   0.010          0.878       0.865       0.061
   0.032          0.929       0.912       0.042
   0.100          0.957       0.935       0.043
   0.316          0.965       0.947       0.051
   1.000          0.971       0.953       0.040
   3.162          0.978       0.941       0.037
  10.000          0.978       0.941       0.032
  31.623          0.985       0.941       0.032
 100.000          0.997       0.947       0.022
 316.228          1.000       0.947       0.022
1000.000          1.000       0.947       0.022
melhor: C = 1 · recall 0.953


**Simulação: a regra de um erro-padrão.** A busca escolheu `C = 1`, mas a figura sugere um platô. Aplique a **regra de um erro-padrão** (Breiman et al., 1984): calcule o erro-padrão da melhor média (desvio entre folds dividido por $\sqrt{5}$), veja quantas configurações caem dentro dele e fique com a mais simples — aqui, o `C` mais regularizado.

In [8]:
i = grid.best_index_
se = table.loc[i, 'desvio val'] / np.sqrt(5)
inside = table[table['recall val'] >= table.loc[i, 'recall val'] - se]
print(f"melhor {table.loc[i, 'recall val']:.3f} · erro-padrão {se:.3f} · {len(inside)} configurações dentro")
print(f"regra de um erro-padrão: C = {inside['C'].min():g}")

melhor 0.953 · erro-padrão 0.018 · 9 configurações dentro
regra de um erro-padrão: C = 0.1


**Para fechar a parte fundamental.** Em uma frase cada: (1) o que a validação cruzada avalia, um modelo ou uma receita? (2) Por que o desvio entre folds precisa ser reportado junto com a média? (3) O que acontece se o `StandardScaler` for ajustado antes do `cross_val_score`?

## Sua vez (fundamental)

Para casa:

1. **Quantos folds?** Repita o passo 4 com `cv=StratifiedKFold(3, ...)` e com `10`. O que acontece com a média e com o desvio entre folds? Quanto custa em tempo de ajuste?
2. **Sem estratificar.** Troque o `StratifiedKFold` do passo 3 por `KFold(5, shuffle=True, random_state=42)` e olhe quantos malignos caem em cada fold de validação (`for _, va in cv.split(X_train, y_train): print(y_train.iloc[va].sum())`). Com 37% de positivos a diferença é pequena; refaça imaginando 3% e explique por que a estratificação viraria obrigatória.
3. **Fora do pipeline.** Ajuste um `StandardScaler` em todo o `X_train`, transforme e rode o `cross_val_score` só com a `LogisticRegression`. Compare com o passo 4. A diferença é grande aqui? Por que você não pode concluir daí que escalonar fora do pipeline é seguro?

## Passo 7 — ajustar os candidatos nos mesmos folds

**Preveja.** Cada família ganha a própria grade. Por que as duas precisam usar o **mesmo** objeto `cv`? O que você mediria se ajustasse a logística e deixasse o k-NN no padrão?

In [9]:
from sklearn.neighbors import KNeighborsClassifier

knn = make_pipeline(StandardScaler(), KNeighborsClassifier())
search_lr = GridSearchCV(model, {'logisticregression__C': np.logspace(-3, 3, 13),
                                 'logisticregression__class_weight': [None, 'balanced']},
                         cv=cv, scoring=['recall', 'roc_auc'], refit='recall')
search_knn = GridSearchCV(knn, {'kneighborsclassifier__n_neighbors': [1, 3, 5, 7, 9, 11, 15, 21, 31],
                                'kneighborsclassifier__weights': ['uniform', 'distance']},
                          cv=cv, scoring=['recall', 'roc_auc'], refit='recall')

for name, search in [('logística', search_lr), ('k-NN', search_knn)]:
    search.fit(X_train, y_train)
    i = search.best_index_
    best_params = ', '.join(f"{k.split('__')[1]}={v:.3g}" if isinstance(v, float) else f"{k.split('__')[1]}={v}"
                            for k, v in search.best_params_.items())
    print(f"{name:9s} {len(search.cv_results_['params'])} configurações · melhor: {best_params}")
    print(f"{'':9s} recall {search.best_score_:.3f} · ROC AUC {search.cv_results_['mean_test_roc_auc'][i]:.3f}")

logística 26 configurações · melhor: C=0.316, class_weight=balanced
          recall 0.959 · ROC AUC 0.995


k-NN      18 configurações · melhor: n_neighbors=3, weights=uniform
          recall 0.935 · ROC AUC 0.989


## Passo 8 — comparar tudo nos mesmos folds

**Preveja.** A logística ajustada está à frente do k-NN ajustado por cerca de 0,024 de recall e os desvios entre folds são 0,030 e 0,039. Isso basta para declarar um vencedor?

In [10]:
candidates = {'dummy': dummy,
              'logistic, default': model,
              'logistic, tuned': search_lr.best_estimator_,
              'k-NN, default': knn,
              'k-NN, tuned': search_knn.best_estimator_}
scores = {name: cross_validate(est, X_train, y_train, cv=cv, scoring=['recall', 'roc_auc'])
          for name, est in candidates.items()}
summary = pd.DataFrame({name: {'recall': s['test_recall'].mean(), 'recall std': s['test_recall'].std(),
                               'ROC AUC': s['test_roc_auc'].mean(), 'AUC std': s['test_roc_auc'].std()}
                        for name, s in scores.items()}).T
print(summary.round(3))

                   recall  recall std  ROC AUC  AUC std
dummy               0.000       0.000    0.500    0.000
logistic, default   0.953       0.040    0.996    0.005
logistic, tuned     0.959       0.030    0.995    0.005
k-NN, default       0.918       0.034    0.987    0.014
k-NN, tuned         0.935       0.039    0.989    0.013


## Passo 9 — diferenças pareadas e o vencedor

**Preveja.** Em vez de comparar duas médias, compare fold a fold. Você espera que a logística vença em todos os cinco folds ou que o sinal da diferença troque?

In [11]:
a, b = scores['logistic, tuned'], scores['k-NN, tuned']
d_recall = a['test_recall'] - b['test_recall']
d_auc = a['test_roc_auc'] - b['test_roc_auc']
print(pd.DataFrame({'recall logística': a['test_recall'], 'recall k-NN': b['test_recall'],
                    'tumores': np.round(d_recall * 34).astype(int),   # 34 malignos por fold de validação
                    'diferença AUC': d_auc}).round(3))
for name, d in [('recall', d_recall), ('ROC AUC', d_auc)]:
    print(f"{name:7s} diferença: média {d.mean():+.3f} · desvio {d.std():.3f} · "
          f"logística à frente em {(d > 0).sum()} de 5 folds")

   recall logística  recall k-NN  tumores  diferença AUC
0             0.912        0.941       -1         -0.003
1             0.971        1.000       -1         -0.001
2             1.000        0.912        3          0.003
3             0.941        0.882        2          0.020
4             0.971        0.941        1          0.013
recall  diferença: média +0.024 · desvio 0.047 · logística à frente em 3 de 5 folds
ROC AUC diferença: média +0.006 · desvio 0.009 · logística à frente em 3 de 5 folds


**Simulação: mais folds mudam o veredito?** Cinco folds é pouco. Refaça a comparação pareada com `RepeatedStratifiedKFold(5, n_repeats=10)`, que dá 50 pares, e conte em quantos cada modelo ficou à frente. O empate do passo 9 se mantém?

In [12]:
rcv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)
r = {n: cross_val_score(candidates[n], X_train, y_train, cv=rcv, scoring='recall')
     for n in ['logistic, tuned', 'k-NN, tuned']}
d = r['logistic, tuned'] - r['k-NN, tuned']
print(f"{len(d)} pares · logística à frente em {(d > 0).sum()} · k-NN em {(d < 0).sum()} · empate em {(d == 0).sum()}")
print(f"diferença média {d.mean():+.3f} ± {d.std():.3f}")

50 pares · logística à frente em 29 · k-NN em 6 · empate em 15
diferença média +0.029 ± 0.041


## Passo 10 — teste de sanidade: embaralhe os rótulos

**Preveja.** Com os rótulos embaralhados, não há nada a aprender. Você espera que o recall caia para zero, para 0,5 ou para algum outro valor? (Dica: o vencedor usa `class_weight='balanced'`.)

In [13]:
from sklearn.model_selection import permutation_test_score

best = search_lr.best_estimator_                 # o vencedor do passo 9
score, perm_scores, p_value = permutation_test_score(
    best, X_train, y_train, cv=cv, scoring='recall', n_permutations=100, random_state=0)
print(f"rótulos reais: {score:.3f}")
print(f"rótulos embaralhados: média {perm_scores.mean():.3f}, melhor de 100 {perm_scores.max():.3f}")
print(f"p-valor: {p_value:.4f}")

rótulos reais: 0.959
rótulos embaralhados: média 0.480, melhor de 100 0.582
p-valor: 0.0099


## Passo 11 — previsões fora do fold

**Preveja.** Por que uma matriz de confusão feita com `cross_val_predict` é honesta, enquanto uma feita com `best.predict(X_train)` não é?

In [14]:
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import cross_val_predict

oof = cross_val_predict(best, X_train, y_train, cv=cv)      # cada linha prevista por um modelo que nunca a viu
print(confusion_matrix(y_train, oof, labels=[1, 0]))

[[163   7]
 [  4 281]]


## Passo 12 — o único teste

**Preveja.** A CV disse recall 0,959 ± 0,030. O teste vai cair dentro dessa faixa? Com 42 malignos no teste, quanto vale um tumor em pontos de recall?

In [15]:
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score

pred = best.predict(X_test)                 # o GridSearchCV já reajustou o vencedor em todo o X_train
proba = best.predict_proba(X_test)[:, 1]
print(f"teste: acurácia {accuracy_score(y_test, pred):.3f} · recall {recall_score(y_test, pred):.3f}"
      f" · ROC AUC {roc_auc_score(y_test, proba):.3f}")
print(confusion_matrix(y_test, pred, labels=[1, 0]))

teste: acurácia 0.982 · recall 0.976 · ROC AUC 0.997
[[41  1]
 [ 1 71]]


**Simulação: quanto o próprio ajuste infla o número?** O `best_score_` de 0,959 é o melhor de 26 configurações, medidas nos mesmos folds que o escolheram. Avalie **a busca da logística** (as 26 configurações) com validação cruzada aninhada — `cross_val_score(search_lr, ...)` num `cv` externo — e compare os três números: busca, CV aninhada e teste.

In [16]:
nested = cross_val_score(search_lr, X_train, y_train, cv=StratifiedKFold(5, shuffle=True, random_state=0))
print(f"busca (best_score_) {search_lr.best_score_:.3f}")
print(f"CV aninhada         {nested.mean():.3f} ± {nested.std():.3f}")
print(f"teste               {recall_score(y_test, pred):.3f}")

busca (best_score_) 0.959
CV aninhada         0.965 ± 0.029
teste               0.976


## Passo 13 — o que o modelo aprendeu?

**Preveja.** Os atributos estão padronizados, então os coeficientes são comparáveis. Você espera que os maiores sejam medidas de tamanho, de textura ou de forma? E eles seriam estáveis se o modelo fosse reajustado em outro fold?

In [17]:
coef = pd.Series(best[-1].coef_[0], index=X.columns)
print(coef.sort_values(key=abs, ascending=False).head(6).round(2))

worst texture          0.96
radius error           0.80
worst symmetry         0.72
worst radius           0.68
mean concave points    0.64
worst area             0.64
dtype: float64


**Para fechar a aula.** Em uma frase cada: (1) qual dos cinco vazamentos nenhum esquema de CV consegue pegar? (2) Por que comparar fold a fold enxerga diferenças que duas médias ± desvio escondem? (3) Por que o `best_score_` de uma busca em grade é otimista?

## Sua vez (avançado)

Para casa, com mais tempo:

1. **Limiar.** Use `cross_val_predict(best, X_train, y_train, cv=cv, method='predict_proba')[:, 1]` para obter probabilidades fora do fold. Encontre o maior limiar que dá recall ≥ 0,98 e reporte a precisão que você paga por ele. Depois aplique esse limiar, uma vez, ao conjunto de teste.
2. **Penalidade L1.** Troque por `LogisticRegression(l1_ratio=1, solver='liblinear')` (no scikit-learn anterior à 1.8, escreva `penalty='l1'`) e repita o passo 6. Quantos dos 30 coeficientes são exatamente zero no melhor `C`? O recall muda mais de um desvio-padrão?
3. **A escolha inteira, aninhada.** A simulação do passo 12 aninhou só a busca da logística. Escreva o laço externo você mesmo, como na [seção de CV aninhada](https://insper.github.io/ML/pt/2026.2/classes/validation/#a-maldicao-do-vencedor-de-novo-validacao-cruzada-aninhada), para que ele avalie os passos 7 a 9 juntos: as duas buscas *e* a escolha entre elas. O número cai mais?
4. **Mais candidatos.** Acrescente à tabela do passo 8 um candidato SMOTE + k-NN (um pipeline do `imblearn`, como na [aula de ROC e dados desbalanceados](https://insper.github.io/ML/pt/2026.2/classes/roc-imbalanced/)) ajustado no mesmo `cv` e compare-o com o vencedor fold a fold. Depois da [aula de SVM](https://insper.github.io/ML/pt/2026.2/classes/svm/), volte e acrescente também um `SVC`. Algum deles vence a regressão logística ajustada por mais que a dispersão das diferenças pareadas?